In [7]:
from torch_geometric.nn import GCNConv
import torch
import pickle

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import train_test_split_edges, negative_sampling
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score

from ast import literal_eval
import numpy as np

from torch_geometric.nn import Node2Vec

import os
import torch
import pickle

import warnings
warnings.filterwarnings('ignore')

In [8]:
class GAE(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GAE, self).__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels)
        self.conv2 = GCNConv(2 * out_channels, out_channels)

    def encode(self, x, edge_index, edge_weight):
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        return self.conv2(x, edge_index, edge_weight)

    def decode(self, z, pos_edge_index, neg_edge_index):
        pos_pred = (z[pos_edge_index[0].long()] * z[pos_edge_index[1].long()]).sum(dim=1)
        neg_pred = (z[neg_edge_index[0].long()] * z[neg_edge_index[1].long()]).sum(dim=1)
        return pos_pred, neg_pred

    def forward(self, data):
        z = self.encode(data.x, data.train_pos_edge_index, data.train_pos_edge_weight)
        return z

In [9]:
def load_model():
    # Define the folder for loading the model
    model_folder = "gae-model-v1"

    # Load the GAE model
    model_path = os.path.join(model_folder, "model.pth")
    encoders_path = os.path.join(model_folder, "encoders_metadata.pkl")
    data_path = os.path.join(model_folder, "graph_data.pkl")

    # Load encoders and metadata
    with open(encoders_path, "rb") as f:
        encoders_metadata = pickle.load(f)
        le_job_title = encoders_metadata["le_job_title"]
        le_skills = encoders_metadata["le_skills"]
        le_category = encoders_metadata["le_category"]
        num_node_features = encoders_metadata["num_node_features"]

    print("Encoders and metadata loaded successfully.")

    # Load the graph data object
    with open(data_path, "rb") as f:
        loaded_data = pickle.load(f)
    print("Graph data loaded successfully.")

    # Initialize and load the GAE model
    loaded_model = GAE(num_node_features, 16)  # Ensure architecture matches
    loaded_model.load_state_dict(torch.load(model_path))
    loaded_model.eval()
    print("Model loaded successfully.")

    return loaded_model, loaded_data, le_job_title, le_skills, le_category, num_node_features

In [10]:
def load_data():
    random_seed = 62
    np.random.seed(random_seed)

    job_descriptions = pd.read_csv('./data/processed/job_descriptions_processed-v5.csv')
    resumes = pd.read_csv('./data/processed/general-resume-dataset-processed-v5.csv', converters={'skills': literal_eval})

    job_descriptions = job_descriptions.sample(frac=1, random_state=random_seed).head(20000)

    job_descriptions['skills'] = job_descriptions['skills'].apply(literal_eval)

    job_descriptions['job_title'].fillna('unknown', inplace=True)
    resumes['job_title'].fillna('unknown', inplace=True)
    resumes['category'].fillna('unknown', inplace=True)

    job_descriptions['job_id'] = range(1, len(job_descriptions) + 1)
    resumes['candidate_id'] = range(1, len(resumes) + 1)

    all_titles = job_descriptions['job_title'].tolist() + resumes['job_title'].tolist()
    all_titles.append('unknown')
    all_categories = resumes['category'].tolist()
    all_categories.append('unknown')

    le_job_title = LabelEncoder()
    le_category = LabelEncoder()
    le_job_title.fit(all_titles)
    le_category.fit(all_categories)

    job_descriptions['job_title'] = le_job_title.transform(job_descriptions['job_title'])
    resumes['job_title'] = le_job_title.transform(resumes['job_title'])
    resumes['category'] = le_category.transform(resumes['category'])

    all_skills = set(skill for skills in job_descriptions['skills'].tolist() + resumes['skills'].tolist() for skill in skills)
    le_skills = {skill: i for i, skill in enumerate(all_skills)}

    nodes = []
    edges = []
    node_features = []

    jobs_from_edges = []
    candidates_from_edges = []
    jobs_and_candidates_from_edges = []

    skill_weight_multiplier = 125
    title_weight = 200

    for i, row in job_descriptions.iterrows():
        nodes.append(row['job_id'])
        skills_vector = [0] * len(le_skills)
        if row['skills']:
            for skill in row['skills']:
                skills_vector[le_skills[skill]] = 1
        node_features.append([row['job_title']] + skills_vector)

    for i, row in resumes.iterrows():
        nodes.append(row['candidate_id'] + len(job_descriptions))
        skills_vector = [0] * len(le_skills)
        if row['skills']:
            for skill in row['skills']:
                skills_vector[le_skills[skill]] = 1
        node_features.append([row['job_title']] + skills_vector)

    job_descriptions['skills'] = job_descriptions['skills'].apply(set)
    resumes['skills'] = resumes['skills'].apply(set)
    return job_descriptions, resumes

In [11]:
def predict_for_new_job_v1(new_job_desc, model, data, le_job_title, le_skills, le_category, resumes, job_descriptions, skill_weight=125, title_weight=200, k=5):
    """
    Predict top candidates for a new job description using normalized similarity scores.

    Args:
        new_job_desc (dict): New job description with 'job_title' and 'skills'.
        model (GAE): Trained GAE model.
        data (Data): PyTorch Geometric Data object.
        le_job_title (LabelEncoder): Encoder for job titles.
        le_skills (dict): Encoder for skills.
        le_category (LabelEncoder): Encoder for categories.
        resumes (DataFrame): Candidate data.
        job_descriptions (DataFrame): Job data.
        skill_weight (float): Weight for normalized skill overlap.
        title_weight (float): Weight for title match.
        k (int): Number of top candidates to return.

    Returns:
        DataFrame: Top-k candidates with match details.
    """
    # Encode job title and skills
    try:
        job_title = le_job_title.transform([new_job_desc['job_title']])[0]
    except ValueError:
        job_title = le_job_title.transform(['unknown'])[0]

    skills_vector = [0] * len(le_skills)
    for skill in new_job_desc['skills']:
        if skill in le_skills:
            skills_vector[le_skills[skill]] = 1

    new_job_features = torch.tensor([job_title] + skills_vector, dtype=torch.float).unsqueeze(0)

    # Project or truncate the new job features to match data.x dimensions
    if new_job_features.size(1) > data.x.size(1):  # Truncate if too large
        new_job_features_projected = new_job_features[:, :data.x.size(1)]
    else:  # Pad if too small
        padding = torch.zeros((1, data.x.size(1) - new_job_features.size(1)))
        new_job_features_projected = torch.cat([new_job_features, padding], dim=1)

    new_job_features_projected = new_job_features_projected.to(data.x.device)

    # Append the new node to the feature matrix
    updated_x = torch.cat([data.x, new_job_features_projected], dim=0)

    # Encode the entire graph, including the new job node
    new_node_index = data.x.size(0)
    model.eval()
    with torch.no_grad():
        z = model.encode(updated_x, data.val_pos_edge_index, data.val_pos_edge_weight)
        new_job_embedding = z[new_node_index].unsqueeze(0)
        candidate_embeddings = z[len(data.x) - len(resumes):]

    # Compute similarity scores
    raw_scores = torch.matmul(new_job_embedding, candidate_embeddings.T).cpu().numpy().flatten()

    # Adjust and normalize scores
    predictions = []
    for idx, candidate in resumes.iterrows():
        candidate_id = candidate['candidate_id']
        candidate_job_title = candidate['job_title']
        candidate_skills = candidate['skills']

        # Skill overlap normalization
        job_skills = set(new_job_desc['skills'])
        mutual_skills = job_skills.intersection(candidate_skills)
        normalized_skill_score = len(mutual_skills) / max(len(job_skills), len(candidate_skills))

        # Title similarity
        title_score = 1 if job_title == candidate_job_title else 0

        # Final score calculation
        final_score = raw_scores[idx] + (normalized_skill_score * skill_weight) + (title_score * title_weight)

        predictions.append({
            "Job ID": "New Job",
            "Job Title": le_job_title.inverse_transform([job_title])[0],
            "Candidate ID": candidate_id,
            "Candidate Job Title": le_job_title.inverse_transform([candidate_job_title])[0],
            "Match Percentage": final_score * 100,  # Convert to percentage
            "Mutual Skills": mutual_skills,
            "Job Skills": list(job_skills),
            "Candidate Skills": candidate_skills
        })

    # Convert to DataFrame and sort
    predictions_df = pd.DataFrame(predictions)
    return predictions_df.sort_values(by="Match Percentage", ascending=False).head(k)

In [12]:
loaded_model, loaded_data, le_job_title, le_skills, le_category, num_node_features = load_model()

Encoders and metadata loaded successfully.
Graph data loaded successfully.
Model loaded successfully.


In [13]:
job_descriptions, resumes = load_data()

In [14]:
new_job = {
    "job_title": "data scientist",
    "skills": ["python", "machine learning", "data visualization"]
}

new_job = {
    "job_title": "hr coordinator",
    "skills": ["employee handbook", "development management", "problem solve", "state law", "resource management", "human resource management", "benefit administration", "microsoft office", "customer service", "leadership development"]
}

# Predict top candidates
top_candidates = predict_for_new_job_v1(
    new_job_desc=new_job,
    model=loaded_model,
    data=loaded_data,
    le_job_title=le_job_title,
    le_skills=le_skills,
    le_category=le_category,
    resumes=resumes,
    job_descriptions=job_descriptions,
    k=5
)

# Display the top candidates
top_candidates.head()


,Job ID,Job Title,Candidate ID,Candidate Job Title,Match Percentage,Mutual Skills,Job Skills,Candidate Skills
729,New Job,hr coordinator,730,kindergarten teacher,207169.588216,"{customer service, microsoft office}","[state law, resource management, employee hand...","{administrative skill, differentiate instructi..."
1882,New Job,hr coordinator,1883,executive director,189491.552734,{},"[state law, resource management, employee hand...","{experience design, user experience design, pr..."
1534,New Job,hr coordinator,1535,designer,178961.901855,{},"[state law, resource management, employee hand...","{business economic, toefl, world wide web, fre..."
1999,New Job,hr coordinator,2000,customer service representative,174900.317383,"{customer service, microsoft office}","[state law, resource management, employee hand...","{telephone skill, process improvement, data in..."
198,New Job,hr coordinator,199,digital producer,165061.684946,{microsoft office},"[state law, resource management, employee hand...","{process improvement, production support, info..."
